# Notebook 01 — Modelos LLM e NLP com Hugging Face

**Objetivo:** Demonstrar dominio do ecossistema Hugging Face com tarefas NLP aplicadas ao dominio de bulas medicas.

**Rubrica 1:** Construir aplicacoes NLP com LLMs e ecossistema Hugging Face (5 itens).

## 2.1 Setup e Imports

In [1]:
import osfrom dotenv import load_dotenvload_dotenv()# HF_TOKEN autentica downloads do Hugging Face Hub (evita rate limiting)os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")import torchfrom transformers import pipeline, AutoModel, AutoTokenizer, AutoModelForQuestionAnsweringfrom transformers import AutoModelForSeq2SeqLMfrom scripts.config import DEVICE, NER_MODEL, EMBEDDING_MODEL

PyTorch: 2.6.0+cu124
CUDA disponivel: True
Device configurado: cuda
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
VRAM: 6.4 GB


## 2.2 Carregando Modelo com AutoModel + AutoTokenizer

Demonstracao: `AutoTokenizer`, `AutoModel`, inspecao de hidden states.

**Modelo:** `pucpr/clinicalnerpt-chemical` — BERT para NER clinico em portugues.

In [2]:
model_id = NER_MODEL
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModel.from_pretrained(model_id).to(DEVICE)

inputs = tokenizer("O mecanismo de atencao e poderoso", return_tensors="pt")
inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
outputs = model(**inputs)

print(f"Dimensoes do output: {outputs.last_hidden_state.shape}")
print(f"Interpretacao: [Batch={outputs.last_hidden_state.shape[0]}, Tokens={outputs.last_hidden_state.shape[1]}, Hidden_Dim={outputs.last_hidden_state.shape[2]}]")

tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
print(f"\nTokens: {tokens}")
print(f"Total de tokens: {len(tokens)}")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: pucpr/clinicalnerpt-chemical
Key                 | Status     | 
--------------------+------------+-
classifier.bias     | UNEXPECTED | 
classifier.weight   | UNEXPECTED | 
pooler.dense.bias   | MISSING    | 
pooler.dense.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Dimensoes do output: torch.Size([1, 10, 768])
Interpretacao: [Batch=1, Tokens=10, Hidden_Dim=768]

Tokens: ['[CLS]', 'o', 'mecanismo', 'de', 'at', '##en', '##cao', 'e', 'poderoso', '[SEP]']
Total de tokens: 10


**Observacoes:**
- Tokenizador BERT usa **WordPiece**: palavras frequentes → tokens unicos, raras → sub-tokens
- Limite: **512 tokens**
- `last_hidden_state`: embedding contextualizado de cada token

## 2.3 Pipeline: sentiment-analysis em Frases Clinicas

Modelo generico em dominio especializado — demonstrando limitacoes que motivam fine-tuning.

In [3]:
classifier = pipeline("sentiment-analysis")

frases = [
    "O uso concomitante e contraindicado devido ao risco de arritmia fatal.",
    "Nao ha interacoes conhecidas com este medicamento.",
    "Recomenda-se monitoramento da funcao renal durante o tratamento.",
    "A administracao concomitante de Amoxicilina com Metotrexato pode aumentar a toxicidade.",
    "O medicamento e seguro e bem tolerado pela maioria dos pacientes.",
]

for frase in frases:
    resultado = classifier(frase)[0]
    print(f"[{resultado["label"]:>8} | {resultado["score"]:.3f}] {frase}")

[transformers] No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

[NEGATIVE | 0.991] O uso concomitante e contraindicado devido ao risco de arritmia fatal.
[NEGATIVE | 0.983] Nao ha interacoes conhecidas com este medicamento.
[NEGATIVE | 0.905] Recomenda-se monitoramento da funcao renal durante o tratamento.
[NEGATIVE | 0.934] A administracao concomitante de Amoxicilina com Metotrexato pode aumentar a toxicidade.
[POSITIVE | 0.543] O medicamento e seguro e bem tolerado pela maioria dos pacientes.


**Analise:** O modelo classifica por tom emocional, nao por significado clinico. Precisamos de modelos treinados em dominio clinico.

## 2.4 Pipeline: NER com clinicalnerpt-chemical

**NER (token classification):** identifica nomes de medicamentos — principios ativos e nomes comerciais.

In [4]:
ner = pipeline("ner", model=NER_MODEL, aggregation_strategy="simple",
             device=0 if DEVICE == "cuda" else -1)

trecho = (
    "A probenecida reduz a secrecao tubular renal da amoxicilina. "
    "No uso concomitante com amoxicilina, pode haver aumento dos niveis "
    "de amoxicilina no sangue. A administracao concomitante de alopurinol "
    "durante o tratamento com amoxicilina pode aumentar a probabilidade "
    "de reacoes alergicas da pele. Existem casos raros de INR aumentada "
    "em pacientes mantidos com acenocumarol ou varfarina."
)

entidades = ner(trecho)
print("Entidades encontradas:")
for ent in entidades:
    print(f'{ent["word"]:<25} {ent["score"]:>8.3f} [{ent["start"]}:{ent["end"]}]')

unicos = list(set(ent["word"] for ent in entidades))
print(f"\nMedicamentos identificados ({len(unicos)}): {unicos}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Entidades encontradas:
probe                        0.827 [2:7]
##nec                        0.798 [7:10]
##ida                        0.910 [10:13]
amo                          0.968 [48:51]
##xic                        0.972 [51:54]
##ilina                      0.963 [54:59]
amo                          0.985 [85:88]
##xic                        0.984 [88:91]
##ilina                      0.981 [91:96]
al                           0.997 [186:188]
##op                         0.997 [188:190]
##urin                       0.998 [190:194]
##ol                         0.998 [194:196]
amo                          0.986 [222:225]
##xic                        0.978 [225:228]
##ilina                      0.977 [228:233]
ac                           0.994 [357:359]
##eno                        0.994 [359:362]
##cuma                       0.992 [362:366]
##rol                        0.994 [366:369]
var                          0.966 [373:376]
##fari                       0.962 [376:380]
##na    

**Analise:** Modelo identifica corretamente amoxicilina, probenecida, alopurinol, acenocumarol, varfarina. Encoder-only (BERT) ideal para NER. Este sera o primeiro estagio do pipeline RAG.

## 2.5 Pipeline: text-generation com GPT-2 Portugues

**Decoder-only:** geracao autoregressiva token por token. Demonstracao de alucinacao.

In [5]:
gerador = pipeline("text-generation", model="pierreguillou/gpt2-small-portuguese")

prompt = "Interacao entre Amoxicilina e Ibuprofeno:"

r1 = gerador(prompt, max_length=80, do_sample=True, temperature=0.9)
print("=== temperature=0.9 (criativa) ===")
print(r1[0]["generated_text"])

r2 = gerador(prompt, max_length=80, do_sample=True, temperature=0.3, top_k=20)
print("\n=== temperature=0.3 + top_k=20 (conservadora) ===")
print(r2[0]["generated_text"])

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: pierreguillou/gpt2-small-portuguese
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[transformers] Passing `generation_config` together with generation-related arguments=({'top_k', 'do_sample', 'temperature', 'max_length'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=256) and `max_length`(=80) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== temperature=0.9 (criativa) ===
Interacao entre Amoxicilina e Ibuprofeno: com eles, entre outros, é o autor do livro “As Quatro Espanhas”, que já foi traduzido em 15 línguas, com mais de 500 títulos. A obra é uma das mais influentes obras sobre os diferentes campos do judaísmo, especialmente a de seus discípulos e seguidores, que possuem sua própria visão do judaísmo como um todo. A obra foi traduzida para a língua portuguesa pelo cardeal espanhol Eugenio Alfonso López de Montijo, membro da Academia das Ciências de Madrid, que estudou a obra sob o patrocínio do Papa João Paulo II.

Atualmente a língua espanhola está em sua maioria na Espanha, exceto em Andorra, onde a comunidade não representa nenhuma forma de presença. As traduções de Amoxicilina em francês para a língua português são raras e não representam uma única tradição no mundo hebraico.

A história do judeus na Idade Média remonta ao período em que as várias tribos judaicas se uniram para lutar contra as tentativas de supr


=== temperature=0.3 + top_k=20 (conservadora) ===
Interacao entre Amoxicilina e Ibuprofeno:


































































































































































































































































**Analise:** GPT-2 gera texto fluente mas **alucina** informacoes — nao tem conhecimento medico real. Decoder-only (geracao) vs Encoder-only (compreensao).

## 2.6 Pipeline: fill-mask com BERT Portugues

Revela conhecimento latente do modelo ao prever tokens mascarados.

In [6]:
unmasker = pipeline("fill-mask", model=EMBEDDING_MODEL)

print("=== Teste 1: contexto clinico ===")
r1 = unmasker("O uso concomitante de Amoxicilina com Metotrexato e [MASK] devido ao risco de toxicidade.")
for r in r1:
    print(f"  {r["token_str"]:>12} | score={r["score"]:.4f}")

print("\n=== Teste 2: contexto de seguranca ===")
r2 = unmasker("Nao ha interacoes conhecidas. O medicamento e [MASK] para uso.")
for r in r2[:3]:
    print(f"  {r["token_str"]:>12} | score={r["score"]:.4f}")

Loading weights:   0%|          | 0/203 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

=== Teste 1: contexto clinico ===
      indicado | score=0.0852
  interrompido | score=0.0570
      proibido | score=0.0517
      suspenso | score=0.0346
      perigoso | score=0.0328

=== Teste 2: contexto de seguranca ===
        pronto | score=0.3815
      aprovado | score=0.1163
      indicado | score=0.1084


**Analise:** BERT preve 'contraindicado' no contexto de toxicidade, 'seguro' no contexto positivo. Atencao **bidirecional** permite preencher lacunas. Este modelo sera usado para gerar embeddings na Fase 6.

## 2.7 Sumarizacao com BART (via AutoModel)

**Encoder-decoder:** sumarizacao abstrativa. A partir do Transformers 5.x, usamos `AutoModelForSeq2SeqLM` diretamente.

In [7]:
summ_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
summ_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn").to(DEVICE)

texto = (
    "Miopatia pode ocorrer em pacientes que usam Zarator, sendo mais frequentes "
    "naqueles que usam tambem ciclosporina, fibratos, niacina ou antifungicos "
    "azolicos. A administracao concomitante com medicamentos inibidores do "
    "citocromo P450 3A4 (ciclosporina, eritromicina/claritromicina, inibidores "
    "da protease) pode alterar a quantidade de atorvastatina no sangue. Sao "
    "conhecidas interacoes com antiacidos, colestipol, contraceptivos orais, "
    "varfarina, acido fusidico."
)

inputs = summ_tokenizer(texto, max_length=1024, truncation=True, return_tensors="pt").to(DEVICE)
summary_ids = summ_model.generate(inputs["input_ids"], max_length=80, min_length=30,
                                   num_beams=4, early_stopping=True)
resumo = summ_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
print(f"Original ({len(texto.split())} palavras):")
print(texto[:200] + "...")
print(f"\nResumo ({len(resumo.split())} palavras):")
print(resumo)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

C:\workspace\python\projeto-2-modulo-1-pos\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lord_\.cache\huggingface\hub\models--facebook--bart-large-cnn. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

Original (55 palavras):
Miopatia pode ocorrer em pacientes que usam Zarator, sendo mais frequentes naqueles que usam tambem ciclosporina, fibratos, niacina ou antifungicos azolicos. A administracao concomitante com medicamen...

Resumo (25 palavras):
Miopatia pode ocorrer em pacientes que usam Zarator. Tambem ciclosporina, fibratos, niacina ou antifungicos azolicos. A administracao concomitante com medicamentos inibidores do citocromo P450 3A4.


**Analise:** BART e ~4x maior que BERT (406M vs 110M). Treinado em ingles → qualidade limitada em portugues. `model.generate()` da controle sobre beams, early stopping.

## 2.8 Question Answering com BERT em Portugues (via AutoModel)

**QA extrativo:** encontra span de resposta no contexto. A partir do Transformers 5.x, usamos `AutoModelForQuestionAnswering` diretamente.

In [8]:
qa_model_id = "pierreguillou/bert-base-cased-squad-v1.1-portuguese"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_id)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_id).to(DEVICE)

contexto = (
    "A probenecida reduz a secrecao tubular renal da amoxicilina. "
    "A administracao concomitante de alopurinol durante o tratamento "
    "com amoxicilina pode aumentar a probabilidade de reacoes alergicas "
    "da pele. Existem casos raros de INR aumentada em pacientes mantidos "
    "com acenocumarol ou varfarina, ao receberem um curso de tratamento "
    "com amoxicilina. Se a coadministracao e necessaria, o tempo de "
    "protrombina ou INR deve ser cuidadosamente monitorado."
)

perguntas = [
    "Quais medicamentos interagem com Amoxicilina?",
    "Qual o risco de tomar Amoxicilina com Varfarina?",
    "Qual a dose maxima recomendada de Amoxicilina?",
]

for i, pergunta in enumerate(perguntas, 1):
    inputs = qa_tokenizer(pergunta, contexto, max_length=512,
                          truncation=True, return_tensors="pt").to(DEVICE)
    outputs = qa_model(**inputs)
    start_idx = outputs.start_logits.argmax()
    end_idx = outputs.end_logits.argmax()
    answer_ids = inputs["input_ids"][0][start_idx:end_idx+1]
    answer = qa_tokenizer.decode(answer_ids)
    score = (outputs.start_logits.max() + outputs.end_logits.max()).item()
    print(f"P{i}: {answer} (score: {score:.1f})")

tokenizer_config.json:   0%|          | 0.00/494 [00:00<?, ?B/s]

C:\workspace\python\projeto-2-modulo-1-pos\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lord_\.cache\huggingface\hub\models--pierreguillou--bert-base-cased-squad-v1.1-portuguese. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


vocab.txt:   0%|          | 0.00/210k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/433M [00:00<?, ?B/s]

C:\workspace\python\projeto-2-modulo-1-pos\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lord_\.cache\huggingface\hub\models--neuralmind--bert-base-portuguese-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

P1: acenocumarol ou varfarina (score: 9.0)
P2: INR aumentada (score: 13.4)
P3: probenecida reduz a secrecao tubular renal da amoxicilina. A administracao concomitante de alopurinol (score: -8.9)


**Analise:** P1 extrai medicamentos corretamente. P2 encontra 'INR aumentada' (associacao indireta). P3 retorna algo com score baixo — QA extrativo **sempre retorna um span**, mesmo sem resposta. **Licao para o RAG:** usaremos geracao fundamentada (LLM + contexto), que pode dizer 'sem informacao'.

## 2.9 Tabela Comparativa de Modelos e Arquiteturas

| Modelo | Arquitetura | Param. | Tarefa | Limite | Dominio |
|---|---|---|---|---|---|
| `clinicalnerpt-chemical` | BERT (encoder-only) | 110M | NER | 512 | Clinico PT |
| `biobertpt-all` | BERT (encoder-only) | 110M | Classificacao | 512 | Biomedico PT |
| `bart-large-cnn` | BART (encoder-decoder) | 406M | Sumarizacao | 1024 | Generico EN |
| `gpt2-small-portuguese` | GPT-2 (decoder-only) | 124M | Geracao | 1024 | Generico PT |
| `bert-base-portuguese-cased` | BERT (encoder-only) | 110M | Fill-mask / Embeddings | 512 | Generico PT |

### Diferenças entre Arquiteturas

- **Encoder-only (BERT):** Atencao bidirecional — cada token ve contexto completo. Ideal para compreensao: NER, classificacao, QA extrativo, embeddings.
- **Decoder-only (GPT-2):** Atencao unidirecional/causal — cada token so ve contexto anterior. Ideal para geracao: chatbots, completamento de texto.
- **Encoder-decoder (BART):** Combina ambos — encoder processa entrada, decoder gera saida. Ideal para traducao e sumarizacao.

### Pipeline vs Inferencia Manual

- `pipeline()`: rapido, encapsula tokenizacao + modelo + post-processamento. Ideal para prototipagem.
- Inferencia manual (`AutoModel` + `tokenizer`): controle total sobre tokenizacao, GPU, batched inference. Necessario para fine-tuning e producao.
- A partir do Transformers 5.x, alguns pipelines foram descontinuados (ex: `summarization`, `question-answering`) — usar `AutoModel` diretamente.

## 2.10 Conclusao: O que Aprendemos

### Tarefas uteis para o detector de interacoes:

| Tarefa | Aplicacao no Projeto | Fase |
|---|---|---|
| **NER** | Extrair medicamentos da consulta do usuario | Fase 8 (RAG) |
| **Classificacao** | Classificar interacao (0/1/2) com BioBERTpt fine-tuned | Fase 4 |
| **Embeddings** | Indexar chunks no ChromaDB para busca vetorial | Fase 6 |
| **Geracao (LLM)** | Produzir resposta final fundamentada nos chunks | Fase 8 (RAG) |

### Proximos passos:
- **Fase 3:** Anotar dataset de treino (~1.500 pares) com weak supervision
- **Fase 4:** Fine-tuning do BioBERTpt para classificacao de interacoes
- **Fase 5:** Prompt engineering com LLMs (zero-shot, few-shot, CoT)